# Noise-aware capillary metabolomics demo

Portfolio notebook based on the analytical structure of my Master's thesis project on capillary blood metabolomics and acute exercise response.

**Important:** the original biomedical dataset cannot be shared publicly because it contains human-participant data and was produced in a research collaboration.  
This notebook therefore uses a **synthetic dataset** to demonstrate the same methodological workflow without exposing private or unpublished data.

## What this notebook demonstrates

- repeated-baseline design;
- baseline reliability analysis;
- PCA-based exploration of resting vs post-exercise samples;
- conventional paired post-exercise testing;
- signal-to-noise ranking of metabolite responses;
- participant-level determinant analysis.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Make the notebook robust whether it is run from the repository root
# or directly from the notebooks/ folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURE_DIR = PROJECT_ROOT / "figures" / "demo_outputs"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)

## 1. Synthetic dataset

The original thesis dataset included repeated baseline capillary blood samples and an immediate post-exercise sample.  
Here, we simulate a comparable structure:

- 100 subjects;
- 52 metabolites;
- two resting baseline samples;
- one post-exercise sample;
- participant-level covariates.

The synthetic data are designed to mimic the *analytical problem*, not to reproduce the real study results.

In [ ]:
def simulate_metabolomics_study(
    n_subjects=100,
    n_amino=20,
    n_amino_related=23,
    n_fatty=8,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    # Participant-level metadata
    subject_ids = [f"S{i:03d}" for i in range(1, n_subjects + 1)]
    age = np.clip(rng.normal(27, 8, n_subjects), 18, 63)
    sex_num = rng.binomial(1, 0.68, n_subjects)  # 1 = male, 0 = female
    bmi = np.clip(rng.normal(24.3, 3.5, n_subjects), 18, 36)
    fasted = rng.binomial(1, 0.21, n_subjects)
    not_exhausted = rng.binomial(1, 0.23, n_subjects)

    vo2peak = (
        48
        + 5.5 * sex_num
        - 0.35 * age
        - 0.85 * (bmi - 23)
        - 3.0 * not_exhausted
        + rng.normal(0, 4.5, n_subjects)
    )

    metadata = pd.DataFrame({
        "Subject_ID": subject_ids,
        "Age": age,
        "Sex_num": sex_num,
        "BMI": bmi,
        "Fasted_num": fasted,
        "Not_exhausted_num": not_exhausted,
        "VO2peak": vo2peak,
    })

    # Metabolite names and classes
    metabolite_names = (
        [f"AA_{i:02d}" for i in range(1, n_amino + 1)]
        + [f"AA_related_{i:02d}" for i in range(1, n_amino_related + 1)]
        + [f"FA_{i:02d}" for i in range(1, n_fatty + 1)]
        + ["Lactate"]
    )

    metabolite_class = (
        ["Amino acid"] * n_amino
        + ["Amino-acid related"] * n_amino_related
        + ["Fatty acid"] * n_fatty
        + ["Lactate"]
    )

    class_map = pd.DataFrame({
        "Metabolite": metabolite_names,
        "Class": metabolite_class,
    })

    n_metabolites = len(metabolite_names)

    # Subject-specific metabolic fingerprints.
    subject_effect = rng.normal(0, 0.45, size=(n_subjects, n_metabolites))

    # Baseline concentration level by metabolite.
    metabolite_intercept = rng.normal(5, 0.8, size=n_metabolites)

    # Small baseline measurement noise.
    baseline_noise_sd = rng.uniform(0.04, 0.16, size=n_metabolites)

    baseline_1 = metabolite_intercept + subject_effect + rng.normal(0, baseline_noise_sd, size=(n_subjects, n_metabolites))
    baseline_2 = metabolite_intercept + subject_effect + rng.normal(0, baseline_noise_sd, size=(n_subjects, n_metabolites))

    # Synthetic post-exercise effects by class.
    effect = np.zeros(n_metabolites)
    for j, cls in enumerate(metabolite_class):
        if cls == "Amino acid":
            effect[j] = rng.normal(-0.22, 0.08)
        elif cls == "Amino-acid related":
            effect[j] = rng.normal(-0.18, 0.10)
        elif cls == "Fatty acid":
            effect[j] = rng.normal(0.10, 0.10)
        elif cls == "Lactate":
            effect[j] = 1.35

    # Attenuated response in potentially not-exhausted subjects.
    attenuation = np.where(not_exhausted == 1, 0.65, 1.0)[:, None]

    # Fasting-specific fatty acid component.
    fasting_effect = np.zeros((n_subjects, n_metabolites))
    fatty_idx = np.array([cls == "Fatty acid" for cls in metabolite_class])
    fasting_effect[:, fatty_idx] = fasted[:, None] * rng.normal(0.15, 0.06, size=(1, fatty_idx.sum()))

    post_exercise = (
        metabolite_intercept
        + subject_effect
        + attenuation * effect
        + fasting_effect
        + rng.normal(0, baseline_noise_sd * 1.4, size=(n_subjects, n_metabolites))
    )

    # Long-format dataframe.
    matrices = {
        "T0.1": baseline_1,
        "T0.2": baseline_2,
        "PostExercise": post_exercise,
    }

    rows = []
    for timepoint, matrix in matrices.items():
        df_tp = pd.DataFrame(matrix, columns=metabolite_names)
        df_tp.insert(0, "Timepoint", timepoint)
        df_tp.insert(0, "Subject_ID", subject_ids)
        rows.append(df_tp)

    data = pd.concat(rows, ignore_index=True)
    data = data.merge(metadata, on="Subject_ID", how="left")

    return data, metadata, class_map

data, metadata, class_map = simulate_metabolomics_study()
metabolite_cols = class_map["Metabolite"].tolist()

print("Synthetic long-format data:", data.shape)
print("Number of metabolites:", len(metabolite_cols))
display(data.head())
display(class_map["Class"].value_counts().rename("n_metabolites"))

## 2. Baseline reliability

The repeated-baseline design allows short-term resting variability to be estimated before interpreting the exercise response.

In the original analysis, this step was used to verify whether the capillary metabolome was sufficiently stable to support individual-level response interpretation.

In [ ]:
baseline_1 = data[data["Timepoint"] == "T0.1"].set_index("Subject_ID")[metabolite_cols]
baseline_2 = data[data["Timepoint"] == "T0.2"].set_index("Subject_ID")[metabolite_cols]
post = data[data["Timepoint"] == "PostExercise"].set_index("Subject_ID")[metabolite_cols]

# Overall agreement across all subject-metabolite pairs.
overall_r = np.corrcoef(baseline_1.to_numpy().ravel(), baseline_2.to_numpy().ravel())[0, 1]

# Metabolite-level baseline correlations.
reliability = []
for metabolite in metabolite_cols:
    r, p = stats.pearsonr(baseline_1[metabolite], baseline_2[metabolite])
    abs_diff = (baseline_2[metabolite] - baseline_1[metabolite]).abs().mean()
    reliability.append({
        "Metabolite": metabolite,
        "Baseline_r": r,
        "Mean_abs_baseline_difference": abs_diff,
    })

reliability = pd.DataFrame(reliability).merge(class_map, on="Metabolite")

print(f"Overall baseline agreement r = {overall_r:.3f}")
display(reliability.sort_values("Baseline_r").head())

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 5.0))
ax.scatter(baseline_1.to_numpy().ravel(), baseline_2.to_numpy().ravel(), s=8, alpha=0.35)
ax.set_xlabel("Baseline 1 synthetic log2 intensity")
ax.set_ylabel("Baseline 2 synthetic log2 intensity")
ax.set_title(f"Overall baseline agreement (r = {overall_r:.3f})")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_baseline_agreement.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
reliability.sort_values("Baseline_r").plot(
    x="Metabolite",
    y="Baseline_r",
    kind="bar",
    ax=ax,
    legend=False
)
ax.axhline(0.6, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_ylabel("Pearson r")
ax.set_title("Metabolite-level baseline reliability")
ax.tick_params(axis="x", labelsize=6)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_metabolite_reliability.png", bbox_inches="tight")
plt.show()

## 3. PCA: resting vs post-exercise structure

PCA provides a simple way to check whether post-exercise samples show a global shift relative to the two repeated baselines.

In [ ]:
X = data[metabolite_cols].copy()
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
scores = pca.fit_transform(X_scaled)

scores_df = data[["Subject_ID", "Timepoint", "Fasted_num", "Not_exhausted_num"]].copy()
scores_df["PC1"] = scores[:, 0]
scores_df["PC2"] = scores[:, 1]

print("Explained variance:", np.round(pca.explained_variance_ratio_ * 100, 2))
display(scores_df.head())

In [ ]:
timepoint_order = ["T0.1", "T0.2", "PostExercise"]
markers = {"T0.1": "o", "T0.2": "s", "PostExercise": "^"}

fig, ax = plt.subplots(figsize=(6.3, 5.0))
for tp in timepoint_order:
    sub = scores_df[scores_df["Timepoint"] == tp]
    ax.scatter(sub["PC1"], sub["PC2"], label=tp, marker=markers[tp], alpha=0.7, s=35)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
ax.set_title("PCA of synthetic baseline and post-exercise samples")
ax.legend(title="Timepoint")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_pca_timepoint.png", bbox_inches="tight")
plt.show()

## 4. Conventional post-exercise testing and signal-to-noise analysis

A conventional analysis asks whether a metabolite changes after exercise.  
The noise-aware analysis asks a complementary question:

> Is the post-exercise response larger than the short-term resting variability observed between the two baseline samples?

In [ ]:
def paired_exercise_tests(baseline, post):
    rows = []
    for metabolite in baseline.columns:
        stat, p = stats.ttest_rel(post[metabolite], baseline[metabolite])
        log2_fc = (post[metabolite] - baseline[metabolite]).mean()
        rows.append({
            "Metabolite": metabolite,
            "mean_log2FC": log2_fc,
            "p_value": p,
        })
    out = pd.DataFrame(rows)
    out["q_value"] = multipletests(out["p_value"], method="fdr_bh")[1]
    return out

baseline_mean = (baseline_1 + baseline_2) / 2

conventional = paired_exercise_tests(baseline_1, post)

snr_rows = []
for metabolite in metabolite_cols:
    exercise_reactivity = (post[metabolite] - baseline_mean[metabolite]).abs().mean()
    methodological_noise = (baseline_2[metabolite] - baseline_1[metabolite]).abs().mean()
    snr_rows.append({
        "Metabolite": metabolite,
        "exercise_reactivity": exercise_reactivity,
        "methodological_noise": methodological_noise,
        "SNR": exercise_reactivity / methodological_noise,
    })

snr = pd.DataFrame(snr_rows)
response_results = (
    conventional
    .merge(snr, on="Metabolite")
    .merge(class_map, on="Metabolite")
    .sort_values("SNR", ascending=False)
)

response_results["significant_FDR_0.05"] = response_results["q_value"] < 0.05
response_results["response_exceeds_noise"] = response_results["SNR"] >= 1.0

display(response_results.head(12))
print("FDR-significant metabolites:", int(response_results["significant_FDR_0.05"].sum()))
print("Metabolites with SNR >= 1:", int(response_results["response_exceeds_noise"].sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
plot_df = response_results.sort_values("SNR", ascending=False).copy()

class_colors = {
    "Amino acid": "#4C78A8",
    "Amino-acid related": "#F58518",
    "Fatty acid": "#54A24B",
    "Lactate": "#B279A2",
}

colors = plot_df["Class"].map(class_colors)
ax.bar(plot_df["Metabolite"], plot_df["SNR"], color=colors)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.axhline(1.5, color="gray", linestyle=":", linewidth=1)
ax.axhline(2.0, color="gray", linestyle=":", linewidth=1)
ax.set_ylabel("Signal-to-noise ratio")
ax.set_title("Noise-aware ranking of synthetic exercise responses")
ax.tick_params(axis="x", labelsize=6, rotation=90)

handles = [plt.Line2D([0], [0], marker="s", linestyle="", color=color, label=label)
           for label, color in class_colors.items()]
ax.legend(handles=handles, title="Metabolite class", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_signal_to_noise.png", bbox_inches="tight")
plt.show()

In [ ]:
volcano_df = response_results.copy()
volcano_df["neg_log10_q"] = -np.log10(volcano_df["q_value"].clip(lower=1e-300))

fig, ax = plt.subplots(figsize=(6.2, 5.0))
for cls, sub in volcano_df.groupby("Class"):
    ax.scatter(
        sub["mean_log2FC"],
        sub["neg_log10_q"],
        label=cls,
        alpha=0.75,
        s=45,
        color=class_colors.get(cls)
    )

ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=1)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean post-exercise log2 fold-change")
ax.set_ylabel("-log10(FDR q-value)")
ax.set_title("Conventional paired testing: post-exercise vs baseline")
ax.legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_volcano_post_exercise.png", bbox_inches="tight")
plt.show()

## 5. Participant-level determinant analysis

In the thesis workflow, participant-level variables were tested as possible determinants of noise-adjusted metabolite responses.

Here we demonstrate a compact version using univariable linear models for metabolites with `SNR >= 1`.

In [ ]:
import statsmodels.api as sm

robust_metabolites = response_results.loc[response_results["SNR"] >= 1.0, "Metabolite"].tolist()

# Continuous noise-adjusted response:
# post-exercise deviation from baseline mean, normalized by metabolite-level baseline noise.
response_matrix = (post[robust_metabolites] - baseline_mean[robust_metabolites])
noise_scale = (baseline_2[robust_metabolites] - baseline_1[robust_metabolites]).abs().mean(axis=0)
response_matrix = response_matrix / noise_scale

determinants = metadata.set_index("Subject_ID").loc[response_matrix.index]
predictors = ["VO2peak", "Age", "BMI", "Sex_num", "Fasted_num", "Not_exhausted_num"]

lm_rows = []
for metabolite in robust_metabolites:
    y_response = response_matrix[metabolite]
    for predictor in predictors:
        X_pred = sm.add_constant(determinants[[predictor]])
        model = sm.OLS(y_response, X_pred, missing="drop").fit()
        lm_rows.append({
            "Metabolite": metabolite,
            "Predictor": predictor,
            "beta": model.params[predictor],
            "p_value": model.pvalues[predictor],
        })

lm_results = pd.DataFrame(lm_rows)
lm_results["q_value"] = multipletests(lm_results["p_value"], method="fdr_bh")[1]

display(lm_results.sort_values("q_value").head(12))

In [ ]:
heatmap_data = (
    lm_results
    .pivot(index="Metabolite", columns="Predictor", values="beta")
    .loc[response_results.set_index("Metabolite").loc[robust_metabolites].sort_values("SNR", ascending=False).index]
)

vmax = np.nanmax(np.abs(heatmap_data.values))
fig, ax = plt.subplots(figsize=(7.2, max(5, 0.18 * len(heatmap_data))))
im = ax.imshow(heatmap_data, aspect="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax)

ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=7)
ax.set_title("Univariable determinant analysis: regression coefficients")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Beta coefficient")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_determinant_heatmap.png", bbox_inches="tight")
plt.show()

## 6. Summary

This notebook is intended as a public, reproducible demonstration of the thesis workflow.

It does **not** reproduce the confidential analyses or the original numerical results.  
Its purpose is to show the methodological structure:

1. estimate baseline variability from repeated resting samples;
2. characterize the post-exercise metabolomic shift;
3. compare exercise response magnitude against baseline-derived noise;
4. test participant-level determinants of response heterogeneity.